In [ ]:
!pip install konlpy

In [ ]:
from konlpy.tag import Okt
import re

In [ ]:
#1. 토큰화(Tokenization)

#텍스트 입력
text='귀여운 덕새가 책을 읽는다'

#형태소 분석기
okt=Okt()

#형태소 분석 (표제어 추출 stem=True)
tokens=okt.morphs(text)

#결과 출력
print('형태소 기준 토큰화 결과:',tokens)

형태소 기준 토큰화 결과: ['귀여운', '덕새', '가', '책', '을', '읽는다']


In [ ]:
#텍스트 입력
text='귀여운 덕새가 책을 읽는다.'

#형태소 분석기
okt=Okt()

#형태소+품사 분석
pos_result=okt.pos(text, stem=True)

#자립형태소 품사 (사전에 독립적으로 존재 가능)
independent_tags=['Noun', 'Verb', 'Adjective', 'Adverb', 'Number', 'Determiner', 'Exclamation']

#결과 분류
independent=[] #자립형태소
dependent=[] #의존형태소

for word, tag in pos_result:
  if tag in independent_tags:
    independent.append((word, tag))
  else:
    dependent.append((word, tag))

#출력
print("자립 형태소:")
for w, t in independent:
  print(f' - {w} ({t})')

print('\n 의존 형태소:')
for w , t in dependent:
  print(f' - {w} ({t})')

자립 형태소:
 - 귀엽다 (Adjective)
 - 덕새 (Noun)
 - 책 (Noun)
 - 읽다 (Verb)

 의존 형태소:
 - 가 (Josa)
 - 을 (Josa)
 - . (Punctuation)


In [ ]:
#2. 정제(Cleaning)

#간단한 불용어 리스트 예시
stopwords=['의','가','이','은','들','는','좀','잘','걍','과',
           '도','를','으로','자','에','와','한','하다']

def clean_korean_text(text):
  #(1)특수문자 제거 (한글+공백만 남김)
  text=re.sub(r'[^ㄱ-ㅎ ㅏ-ㅣ 가-힣\s]','',text)

  #(2)형태소 분석(원형 복원 포함)
  tokens=okt.morphs(text, stem=True)

  #(3)불용어 제거+길이 1 이하 제거
  tokens=[word for word in tokens if word not in stopwords and len(word)>1]

  return tokens

In [ ]:
text='나는 오늘도 열심히 공부를 하고 있다.'
print(clean_korean_text(text))

['오늘', '열심히', '공부', '있다']


In [ ]:
#3. 정규화(Normalization)

okt=Okt()

text='귀여운 덕새가 책을 읽고 있다.'

#(1)형태소 분석+어간 추출(stem=True)
tokens_stem=okt.morphs(text, stem=True)
print('어간 추출:', tokens_stem)

#(2)형태소 분석+원형 유지(stem=False)
tokens_raw=okt.morphs(text, stem=False)
print('원형 유지:',tokens_raw)

어간 추출: ['귀엽다', '덕새', '가', '책', '을', '읽다', '있다', '.']
원형 유지: ['귀여운', '덕새', '가', '책', '을', '읽고', '있다', '.']


In [ ]:
#(3/25) 교수님 예제
#형태소 추출 + 정제 + 형태소 내 명사 추출(정규화)

import re
from konlpy.tag import Okt

# 형태소 분석기
okt = Okt()

# 불용어 리스트 (예시)
stopwords = ['의', '가', '이', '은', '들', '는', '좀', '잘', '과', '도', '를', '으로', '에', '와', '하다']

def preprocess_korean_text(text, extract_nouns=False): #extract_nouns=False->명사만 추출할까요? N
    # 0. 형태소 추출
    text_test = okt.morphs(text)
    print("# 0. 모든 형태소 추출 :", text_test)
    print("------------------------------------------")


    # 1. 한글, 공백만 남기고 모두 제거
    text = re.sub(r"[^ㄱ-ㅎㅏ-ㅣ가-힣\s]", "", text)
    print("# 1. 한글, 공백만 남기고 모두 제거 :", text)
    print("------------------------------------------")

    # 2. 중복 공백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    print("# 2. 중복 공백 제거 : ", text)
    print("------------------------------------------")

    # 3. 형태소 분석 또는 명사 추출
    if extract_nouns:
        tokens = okt.nouns(text)  # 명사만 추출
        print("# 3. 형태소 분석 또는 명사 추출 : ", tokens)
        print("------------------------------------------")

    else:
        tokens = okt.morphs(text, stem=True)  # 일반 형태소 분석 + 원형 복원
        print("# 3. 형태소 분석 또는 명사 추출 : ", tokens)
        print("------------------------------------------")


    # 4. 불용어 제거 + 길이 1 이하 토큰 제거
    tokens = [word for word in tokens if word not in stopwords and len(word) > 1]

    return tokens

In [ ]:
sample_text = "귀여운 덕새가 책을 읽는다. "
tokens1 = preprocess_korean_text(sample_text, extract_nouns=False)
print("형태소 분석 기반 전처리:", tokens1)

# 0. 모든 형태소 추출 : ['귀여운', '덕새', '가', '책', '을', '읽는다', '.']
------------------------------------------
# 1. 한글, 공백만 남기고 모두 제거 : 귀여운 덕새가 책을 읽는다 
------------------------------------------
# 2. 중복 공백 제거 :  귀여운 덕새가 책을 읽는다
------------------------------------------
# 3. 형태소 분석 또는 명사 추출 :  ['귀엽다', '덕새', '가', '책', '을', '읽다']
------------------------------------------
형태소 분석 기반 전처리: ['귀엽다', '덕새', '읽다']


In [ ]:
# 명사 기반 전처리
tokens2 = preprocess_korean_text(sample_text, extract_nouns=True) #명사만 추출할 것->3단계에서 명사만 남음
print("형태소 분석 기반 전처리:", tokens2)

# 0. 모든 형태소 추출 : ['귀여운', '덕새', '가', '책', '을', '읽는다', '.']
------------------------------------------
# 1. 한글, 공백만 남기고 모두 제거 : 귀여운 덕새가 책을 읽는다 
------------------------------------------
# 2. 중복 공백 제거 :  귀여운 덕새가 책을 읽는다
------------------------------------------
# 3. 형태소 분석 또는 명사 추출 :  ['덕새', '책']
------------------------------------------
형태소 분석 기반 전처리: ['덕새']


In [ ]:
#문제1: '귀엽다', '덕새', '읽다' 만 남기기

import re
from konlpy.tag import Okt

# 형태소 분석기
okt = Okt()

#간단한 불용어 리스트 예시
stopwords=['의','가','이','은','들','는','좀','잘',
           '과','도','를','으로','에','와','하다']

def clean_korean_text(text):
  #(1)특수문자 제거 (한글&공백만 남김)
  text=re.sub(r'[^ㄱ-ㅎ ㅏ-ㅣ 가-힣\s]','',text)
  print('1단계: ',text)
  print("------------------------------------------")

  #(2)형태소 분석(원형 복원 O)
  tokens=okt.morphs(text, stem=True)
  print('2단계: ',tokens)
  print("------------------------------------------")

  #(3)불용어 제거 + 길이 1 이하 제거
  tokens=[word for word in tokens if word not in stopwords and len(word)>1]
  print('3단계: ',tokens)
  print("------------------------------------------")

  return tokens

In [ ]:
sample_text = "귀여운 덕새가 책을 읽는다. "
tokens1 = clean_korean_text(sample_text)

1단계:  귀여운 덕새가 책을 읽는다 
------------------------------------------
2단계:  ['귀엽다', '덕새', '가', '책', '을', '읽다']
------------------------------------------
3단계:  ['귀엽다', '덕새', '읽다']
------------------------------------------
